In [1]:
import torch
import numpy as np
## SVD matrix constructions
d = 10
k  = 10
w_rank = 2
w = torch.randn(d, w_rank) @ torch.randn(w_rank, k)
print(w)
print("Rank of the Matrix is: ", np.linalg.matrix_rank(w))


tensor([[-2.9775, -1.0205,  0.8303, -0.7715,  0.5140, -2.1190,  1.9842,  1.6535,
          2.6070, -0.4698],
        [-5.4139, -1.6057,  1.6362, -1.7050,  0.9707, -4.4050,  3.8129,  3.1266,
          4.7216, -0.2375],
        [-4.0943, -1.3339,  1.1769, -1.1447,  0.7168, -3.0670,  2.7853,  2.3070,
          3.5797, -0.4748],
        [-3.6175, -1.2298,  1.0139, -0.9494,  0.6260, -2.5967,  2.4189,  2.0137,
          3.1667, -0.5460],
        [-5.1964, -2.2999,  1.1863, -0.7185,  0.8222, -2.5511,  3.0367,  2.6359,
          4.5886, -2.1012],
        [ 0.2983,  0.0650, -0.1020,  0.1223, -0.0569,  0.2946, -0.2294, -0.1836,
         -0.2584, -0.0449],
        [ 3.0727,  1.5889, -0.5856,  0.1479, -0.4532,  1.0026, -1.6077, -1.4485,
         -2.7305,  1.8076],
        [ 1.0231,  0.6158, -0.1510, -0.0558, -0.1383,  0.1419, -0.4640, -0.4405,
         -0.9156,  0.8162],
        [ 1.8544,  0.7205, -0.4741,  0.3777, -0.3079,  1.1319, -1.1660, -0.9889,
         -1.6300,  0.5024],
        [-2.8097, -

In [2]:
# svd on w where w = UxSxV^T
U, S, V = torch.svd(w)
U_r = U[:, :w_rank]
S_r = torch.diag(S[:w_rank])
# .diag builds a 2D diagonal matrix from a 1D vector or extracts the diagonal elements from a 2D matrix
V_r = V[:, :w_rank].t()
# construct B, A
B = U_r @ S_r
A = V_r
print(B.shape) 
print(A.shape) 

torch.Size([10, 2])
torch.Size([2, 10])


### **LORA** 
We define the LORA parameterization based on the description in the paper

In [ ]:
import torch.nn as nn
class LORA(nn.Module):
    def __init__(self, in_features, out_features, rank =1, alpha =1):
        super().__init__()
        # gaussian initialization of A and a zero matrix (why zero - refer README)
        self.lora_A = nn.Parameter(torch.empty((rank, out_features)))  # rxk
        self.lora_B = nn.Parameter(torch.zeros((in_features, rank)))   # dxr
        nn.init.normal(self.lora_A, mean =0, std =1)
        
        # scale the weights(BxA) by alpha/rank
        self.scale = alpha/rank
        self.enabled = True
    
    def forward(self, model_weights):
        if self.enabled:
            return model_weights + torch.matmul(self.lora_B , self.lora_A).view(model_weights.shape) * self.scale
        else:
            return model_weights
                

The above parameterization can then be added to a network, it requires us to 
- train the model 
- store model's original weights
- register parameterization
- freeze the non lora params, only fine tune those introduced by LoRA
- can be done using:
if params name does'nt have a lora prefix then set 
 `params.requires_grad = False`  
 this way backprop does'nt happen on those params
 - find the target in the data to fine tune to optimize performance for that specific task, then load the train set and target set accordingly

**Extending TO QLoRA**

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class NF4Quantizer:
    # The 16 quantile values derived from a normal distribution N(0, 1) in [-1, 1] range
    NF4_LUT = torch.tensor([
        -1.0, -0.6961, -0.5251, -0.3949, -0.2844, -0.1848, -0.0910, 0.0,
        0.0795, 0.1609, 0.2471, 0.3396, 0.4407, 0.5581, 0.7027, 1.0
    ])

    @classmethod
    def quantize(cls, weight: torch.Tensor, block_size: int = 64):
        out_features, in_features = weight.shape
        flattened = weight.reshape(-1, block_size)

        # Calculates per-block absolute maximum scale
        block_scales = flattened.abs().max(dim=-1, keepdim=True).values
        block_scales = torch.clamp(block_scales, min=1e-8)

        # Normalize weight values to  [-1, 1] 
        normalized_w = flattened / block_scales

        # Finds closest NF4 bucket index (0 to 15) for storing weights
        lut = cls.NF4_LUT.to(weight.device)
        distances = (normalized_w.unsqueeze(-1) - lut).abs()
        q_indices = distances.argmin(dim=-1).to(torch.uint8)

        # Reshape indices back to original weight shape
        q_indices = q_indices.reshape(out_features, in_features)
        block_scales = block_scales.squeeze(-1)

        return q_indices, block_scales

    @classmethod
    def dequantize(cls, q_indices: torch.Tensor, block_scales: torch.Tensor, block_size: int = 64):
        lut = cls.NF4_LUT.to(q_indices.device)
        
        # Look up floating point values for 4-bit indices
        dequantized_norm = lut[q_indices.long()]
        
        # Reshape to re-apply block scales
        dequantized_blocks = dequantized_norm.reshape(-1, block_size)
        original_w = dequantized_blocks * block_scales.unsqueeze(-1)

        return original_w.reshape(q_indices.shape)


class QLoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, base_weight: torch.Tensor, 
                 rank: int = 1, alpha: float = 1.0, block_size: int = 64, bias: torch.Tensor = None):
        super().__init__()
        
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scale = alpha / rank
        self.enabled = True
        self.block_size = block_size

        with torch.no_grad():
            q_indices, block_scales = NF4Quantizer.quantize(base_weight, block_size=self.block_size)
            
            # Register quantized values as buffers so torch does NOT track gradients for them
            self.register_buffer("qweight_nf4", q_indices)
            self.register_buffer("block_scales", block_scales)

            if bias is not None:
                self.register_buffer("bias", bias)
            else:
                self.bias = None


        self.lora_A = nn.Parameter(torch.empty((rank, in_features)))
        nn.init.normal_(self.lora_A, mean=0.0, std=1.0 / math.sqrt(rank))

        # B: (out_features, rank) -> Initialized with zeros so Delta W starts at 0
        self.lora_B = nn.Parameter(torch.zeros((out_features, rank)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base Model : dequantization of NF4 weights
        w0 = NF4Quantizer.dequantize(self.qweight_nf4, self.block_scales, self.block_size).to(x.dtype)
        out = F.linear(x, w0, self.bias)

        # LoRA Branch: Calculate Delta W * x if LoRA is enabled
        if self.enabled:
            # (x @ A.T) @ B.T -> gives output shape (batch_size, seq_len, out_features)
            lora_out = (F.linear(F.linear(x, self.lora_A), self.lora_B)) * self.scale
            out = out + lora_out

        return out